In [ ]:
!git clone https://github.com/luythoangduy/gfad.git

In [ ]:
!pip install hydra-core torchmetrics

In [ ]:
%cd gfad

In [ ]:
!python foundad/src/sample.py source=/kaggle/input/datasets/ipythonx/mvtec-ad target=../mvtec_tmp seed=0 num_samples=2

In [ ]:
from pathlib import Path

cfg_path = Path("/kaggle/working/gfad/foundad/configs/app/train_dinov3.yaml")

cfg_text = """# @package app
meta:
  model: dinov3
  crop_size: 512
  pred_depth: 6
  pred_emb_dim: 384
  use_bfloat16: false
  if_pred_pe: false
  n_layer: 3
  feat_normed: false
  loss_mode: l2
  weights: https://dinov3.llamameta.net/dinov3_vitb16/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth?Policy=eyJTdGF0ZW1lbnQiOlt7InVuaXF1ZV9oYXNoIjoiYWRzZWcycjNiYm9rYngwb3FhbWVtOXg2IiwiUmVzb3VyY2UiOiJodHRwczpcL1wvZGlub3YzLmxsYW1hbWV0YS5uZXRcLyoiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3ODEyODE2NzV9fX1dfQ__&Signature=e4naiZWubLdXDc95VrdpbkyPBhWv5Szz9cyCLKwt6y%7EQszh8q4e4V8Ajpx1I1IctmtFyawhP-3vGAWH9gNdbahVUakA4KlTXo4q2UhGlMPlM6kk8dvRKJ4PNkBbn-D9vuNdaxiFniSrPEN2UcoTVtXLz-7v21iVY7FLOPnlLz3M9EGHb5jSq-t656IMPXui7AUxpqLNgRDHbiRrOGkdY2YaWpakBkpQGZTod33kn6uNX0lrhsAR615uMM-QLsk-R1N3FC0G46ACs6ACTfE5Jeuis8qSH3O6ih8wvbwHtfMe6nhjj8Dx3rGK8oDmnT2R8d60P4n9jgQpFmnQUgNaKZw__&Key-Pair-Id=K15QRJLYKIFSLZ&Download-Request-ID=1549062236932985
  gated_attention:
      enabled: true
      mode: residual
      positions: [attn_output]
      granularity: elementwise
      activation: sigmoid
      init_bias: 0.0

logging:
  folder: logs/${data.data_name}/${app.meta.model}${oc.select:diy_name,""}
  write_tag: train
"""

cfg_path.write_text(cfg_text)
print(cfg_path.read_text())

In [ ]:
!python foundad/main.py mode=train data.batch_size=8 data.dataset=mvtec data.num_workers=4 data.data_name=mvtec_tmp data.data_path=.. app=train_dinov3 diy_name=dbug dist.backend=gloo

In [ ]:
!python foundad/main.py mode=AD data.dataset=mvtec data.data_name=mvtec_tmp diy_name=dbug data.num_workers=4 data.test_root=/kaggle/input/datasets/ipythonx/mvtec-ad app=test app.ckpt_step=2000